# Insurance Premium Prediction - Model Training
**Student ML Project**: Training and comparing multiple models

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import pickle
import warnings
warnings.filterwarnings('ignore')

print('📚 All libraries imported successfully!')
print('Ready to start model training!')

In [ ]:
# Load the insurance dataset
df = pd.read_csv('insurance (1).csv')

print('📊 Dataset loaded successfully!')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Encode categorical variables
print('🔄 Encoding categorical variables...')

encoders = {}
categorical_cols = ['sex', 'smoker', 'region']

for col in categorical_cols:
    encoders[col] = LabelEncoder()
    df[col + '_encoded'] = encoders[col].fit_transform(df[col])
    print(f'   ✅ {col} encoded: {df[col].unique()} -> {df[col + "_encoded"].unique()}')

print('\n🎉 All categorical variables encoded!')

In [ ]:
# Create interaction features
print('🔧 Creating interaction features...')

df['age_bmi'] = df['age'] * df['bmi']
df['smoker_age'] = df['smoker_encoded'] * df['age']
df['smoker_bmi'] = df['smoker_encoded'] * df['bmi']

print('   ✅ age_bmi: Age × BMI interaction')
print('   ✅ smoker_age: Smoker × Age interaction')
print('   ✅ smoker_bmi: Smoker × BMI interaction')
print('\n🎉 Feature engineering complete!')

In [ ]:
# Prepare features and target
print('📋 Preparing features and target variable...')

feature_cols = ['age', 'sex_encoded', 'bmi', 'children', 'smoker_encoded', 
                'region_encoded', 'age_bmi', 'smoker_age', 'smoker_bmi']

X = df[feature_cols]
y = df['charges']

print(f'✅ Features selected: {len(feature_cols)}')
print(f'   Feature names: {feature_cols}')
print(f'✅ X shape: {X.shape}')
print(f'✅ y shape: {y.shape}')

In [ ]:
# Split data into training and testing sets
print('✂️ Splitting data into train and test sets...')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'✅ Training set: {X_train.shape[0]} samples')
print(f'✅ Testing set: {X_test.shape[0]} samples')
print(f'✅ Split ratio: 80% train, 20% test')

print(f'\n📊 Training target statistics:')
print(f'   Mean: ${y_train.mean():.2f}')
print(f'   Min: ${y_train.min():.2f}')
print(f'   Max: ${y_train.max():.2f}')

In [ ]:
# Import all ML models
print('🤖 Importing machine learning models...')

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.ensemble import AdaBoostRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

print('✅ All ML models imported successfully!')

In [ ]:
# Setup all models to test
print('⚙️ Setting up models for training...')

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0, random_state=42),
    'Lasso Regression': Lasso(alpha=1.0, random_state=42),
    'ElasticNet': ElasticNet(alpha=1.0, random_state=42),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'AdaBoost': AdaBoostRegressor(n_estimators=100, random_state=42),
    'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'SVR': SVR(kernel='rbf')
}

print(f'✅ {len(models)} models ready for training:')
for i, name in enumerate(models.keys(), 1):
    print(f'   {i:2d}. {name}')

In [ ]:
# Train and evaluate all models
print('🚀 Training and evaluating all models...')
print('This might take a few minutes depending on your computer...')

# Dictionary to store results
results = {}

# Train and evaluate each model
for name, model in models.items():
    print(f'\n🔄 Training {name}...')
    
    try:
        # Train the model
        model.fit(X_train, y_train)
        
        # Make predictions
        y_pred = model.predict(X_test)
        
        # Calculate metrics
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        
        # Store results
        results[name] = {
            'MSE': mse,
            'R2': r2,
            'MAE': mae,
            'model': model
        }
        
        print(f'   ✅ {name} completed!')
        print(f'      Mean Squared Error: {mse:.4f}')
        print(f'      R² Score: {r2:.4f}')
        print(f'      Mean Absolute Error: {mae:.2f}')
        
    except Exception as e:
        print(f'   ❌ {name} failed: {str(e)}')
        results[name] = {
            'MSE': float('inf'),
            'R2': -999,
            'MAE': float('inf'),
            'model': None
        }

print('\n🎉 All models trained and evaluated!')

In [ ]:
# Display model performance summary
print('📊 Model Performance Summary:')
print('=' * 70)
print(f'{'Model':<20} {'MSE':<12} {'R² Score':<12} {'MAE ($)':<12}')
print('=' * 70)

# Sort results by R² score (descending)
sorted_results = sorted(results.items(), key=lambda x: x[1]['R2'], reverse=True)

for name, metrics in sorted_results:
    if metrics['R2'] > -999:  # Only show successful models
        print(f'{name:<20} {metrics["MSE"]:<12.4f} {metrics["R2"]:<12.4f} {metrics["MAE"]:<12.2f}')
    else:
        print(f'{name:<20} {'FAILED':<12} {'FAILED':<12} {'FAILED':<12}')

print('=' * 70)

In [ ]:
# Find and display the best model
print('🏆 Finding the best model...')

# Find the best model (highest R² score)
valid_results = {k: v for k, v in results.items() if v['R2'] > -999}
best_model_name = max(valid_results, key=lambda x: valid_results[x]['R2'])
best_model = valid_results[best_model_name]['model']
best_r2 = valid_results[best_model_name]['R2']
best_mse = valid_results[best_model_name]['MSE']
best_mae = valid_results[best_model_name]['MAE']

print(f'\n🏆 WINNER: {best_model_name}')
print(f'   🎯 R² Score: {best_r2:.4f} ({best_r2*100:.2f}% accuracy)')
print(f'   📊 MSE: {best_mse:.4f}')
print(f'   💰 MAE: ${best_mae:.2f}')

print(f'\n📈 Top 3 Models:')
for i, (name, metrics) in enumerate(sorted_results[:3], 1):
    if metrics['R2'] > -999:
        print(f'   {i}. {name}: {metrics["R2"]*100:.2f}% accuracy')

# Save the best model for later use
model = best_model
model_type = best_model_name

print(f'\n💾 Best model saved as "model" variable for future use!')

In [ ]:
# Save the best model and related files
print('💾 Saving the best model and encoders...')

# Save the best model
with open('best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print('✅ Best model saved as "best_model.pkl"')

# Save encoders
with open('encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)
print('✅ Encoders saved as "encoders.pkl"')

# Save model information
model_info = {
    'model_name': best_model_name,
    'r2_score': best_r2,
    'mae': best_mae,
    'mse': best_mse,
    'feature_names': feature_cols,
    'training_samples': len(X_train),
    'test_samples': len(X_test)
}

with open('model_info.pkl', 'wb') as f:
    pickle.dump(model_info, f)
print('✅ Model info saved as "model_info.pkl"')

print(f'\n🚀 All files saved! Ready for Streamlit deployment!')
print(f'\n📋 Summary:')
print(f'   🏆 Best Model: {best_model_name}')
print(f'   🎯 Accuracy: {best_r2*100:.2f}%')
print(f'   💰 Average Error: ${best_mae:.2f}')
print(f'   📁 Files: best_model.pkl, encoders.pkl, model_info.pkl')